# 🧠 Emotional Distress Detection Analytics Pipeline

This notebook demonstrates the NLP analytics pipeline for detecting early emotional distress signals from Instagram data.

## System Overview

The pipeline consists of two stages:

1. **Stage 1: NLP Signal Extraction** - Analyzes text (captions/comments) for:
   - Sentiment (positive/negative/neutral)
   - Emotions (sadness, anger, fear, joy, etc.)
   - Cognitive distortions (catastrophizing, hopelessness, etc.)

2. **Stage 2: Behavioral Feature Engineering** - Aggregates signals to compute:
   - Distortion metrics and trends
   - Sentiment volatility
   - Engagement patterns
   - Risk scores and priorities

## Setup
First, ensure you're in the backend directory and have the virtual environment activated.

In [65]:
import os
import sys
import asyncio
from datetime import datetime

# Add backend to path
backend_path = os.path.join(os.getcwd(), 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

print(f"✅ Backend path added: {backend_path}")
print(f"📁 Current directory: {os.getcwd()}")

✅ Backend path added: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/backend
📁 Current directory: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled


## Step 1: Test MongoDB Connection

Let's verify the database connection before running the pipeline.

In [66]:
from config.database import MongoDB
from loguru import logger

async def test_connection():
    """Test MongoDB connection"""
    try:
        await MongoDB.connect_db()
        db = MongoDB.get_db()
        
        # Count documents in collections
        posts_count = await db.instagram_posts.count_documents({})
        users_count = await db.instagram_users.count_documents({})
        
        print("=" * 60)
        print("📊 Database Connection Status")
        print("=" * 60)
        print(f"✅ Connected to MongoDB")
        print(f"📝 Instagram posts: {posts_count}")
        print(f"👤 Instagram users: {users_count}")
        print("=" * 60)
        
        return True
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return False

# Run the test
await test_connection()

2026-02-28 23:22:11.623 | SUCCESS  | config.database:connect_db:26 - Connected to MongoDB database: instagram_scraper


📊 Database Connection Status
✅ Connected to MongoDB
📝 Instagram posts: 164
👤 Instagram users: 11


True

## Step 2: Test Individual NLP Models

Let's test each NLP model independently to ensure they're working correctly.

In [67]:
from analytics.nlp_models import SentimentAnalyzer, EmotionDetector, CognitiveDistortionDetector

# Sample texts for testing
test_texts = [
    "I love this post! So inspiring and beautiful!",  # Positive
    "I always fail at everything. Nothing ever works out.",  # Negative with distortion
    "Feeling really sad today. Nobody understands me.",  # Sadness
    "This is just a regular comment.",  # Neutral
]

print("🧪 Testing NLP Models")
print("=" * 70)

# Test Sentiment Analysis
print("\n📊 Sentiment Analysis:")
print("-" * 70)
sentiment_analyzer = SentimentAnalyzer()

for text in test_texts[:2]:  # Test first 2
    result = sentiment_analyzer.analyze(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Label: {result['label']} (score: {result['score']:.3f})")
    print(f"→ Sentiment score: {result['sentiment_score']:.3f}")

print("\n" + "=" * 70)

2026-02-28 23:22:11.775 | INFO     | analytics.nlp_models:__init__:33 - Loading sentiment model: cardiffnlp/twitter-roberta-base-sentiment-latest


🧪 Testing NLP Models

📊 Sentiment Analysis:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2138.19it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 23:22:14.301 | SUCCESS  | analytics.nlp_models:__init__:45 - Sentiment model loaded on cpu



Text: I love this post! So inspiring and beautiful!...
→ Label: positive (score: 0.985)
→ Sentiment score: 0.979

Text: I always fail at everything. Nothing ever works out....
→ Label: negative (score: 0.891)
→ Sentiment score: -0.871



In [68]:
# Test Emotion Detection
print("\n😊 Emotion Detection:")
print("-" * 70)
emotion_detector = EmotionDetector()

for text in test_texts:
    result = emotion_detector.detect(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Emotion: {result['label']} (confidence: {result['score']:.3f})")
    print(f"→ Distress: {result['is_distress']} (score: {result['distress_score']:.3f})")

print("\n" + "=" * 70)

2026-02-28 23:22:14.451 | INFO     | analytics.nlp_models:__init__:131 - Loading emotion model: j-hartmann/emotion-english-distilroberta-base



😊 Emotion Detection:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2006.92it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 23:22:16.374 | SUCCESS  | analytics.nlp_models:__init__:143 - Emotion model loaded on cpu



Text: I love this post! So inspiring and beautiful!...
→ Emotion: joy (confidence: 0.987)
→ Distress: False (score: 0.003)

Text: I always fail at everything. Nothing ever works out....
→ Emotion: neutral (confidence: 0.329)
→ Distress: False (score: 0.522)

Text: Feeling really sad today. Nobody understands me....
→ Emotion: sadness (confidence: 0.986)
→ Distress: True (score: 0.987)

Text: This is just a regular comment....
→ Emotion: neutral (confidence: 0.965)
→ Distress: False (score: 0.011)



In [69]:
# Test Cognitive Distortion Detection
print("\n🧠 Cognitive Distortion Detection:")
print("-" * 70)
distortion_detector = CognitiveDistortionDetector(threshold=0.55)

for text in test_texts:
    result = distortion_detector.detect(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Distortion detected: {result['distortion_indicator'] == 1}")
    if result['distortion_category']:
        print(f"→ Type: {result['distortion_category']}")
    print(f"→ Score: {result['distortion_score']:.3f}")

print("\n" + "=" * 70)

2026-02-28 23:22:16.513 | INFO     | analytics.nlp_models:__init__:278 - Loading distortion detection model: sentence-transformers/all-MiniLM-L6-v2



🧠 Cognitive Distortion Detection:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1567.54it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 23:22:20.937 | SUCCESS  | analytics.nlp_models:__init__:289 - Cognitive distortion detector initialized



Text: I love this post! So inspiring and beautiful!...
→ Distortion detected: False
→ Score: 0.183

Text: I always fail at everything. Nothing ever works out....
→ Distortion detected: True
→ Type: overgeneralization
→ Score: 0.801

Text: Feeling really sad today. Nobody understands me....
→ Distortion detected: False
→ Score: 0.346

Text: This is just a regular comment....
→ Distortion detected: False
→ Score: 0.244



## Step 3: Run Stage 1 - NLP Signal Extraction

This stage extracts text from Instagram posts/comments and runs all NLP models.

**Note:** This may take several minutes depending on the number of posts in your database.

In [70]:
from analytics.signal_extraction import NLPSignalExtractor

async def run_stage1(case_users=None, limit=10):
    """
    Run Stage 1: NLP Signal Extraction
    
    Args:
        case_users: List of specific usernames (None = all)
        limit: Max posts to process (None = all)
    """
    print("🚀 Starting Stage 1: NLP Signal Extraction")
    print("=" * 70)
    
    extractor = NLPSignalExtractor()
    
    results = await extractor.run_pipeline(
        case_users=case_users,
        limit=limit,
        export_csv=True,
        csv_path="nlp_signals_output.csv"
    )
    
    return results

# Run Stage 1 with a small limit for testing
# Change limit=None to process all posts
stage1_results = await run_stage1(limit=10)

print("\n📊 Stage 1 Results:")
print(f"   • Text units found: {stage1_results['text_units_found']}")
print(f"   • Valid units analyzed: {stage1_results['valid_units']}")
print(f"   • Signals created: {stage1_results['signals_created']}")
print(f"   • CSV exported: {stage1_results['csv_file']}")
print(f"   • Duration: {stage1_results['duration_seconds']:.2f}s")

2026-02-28 23:22:21.079 | INFO     | analytics.signal_extraction:__init__:52 - NLP Signal Extractor initialized
2026-02-28 23:22:21.080 | INFO     | analytics.signal_extraction:run_pipeline:311 - ======================================================================
2026-02-28 23:22:21.081 | INFO     | analytics.signal_extraction:run_pipeline:312 - Starting NLP Signal Extraction Pipeline (Stage 1)
2026-02-28 23:22:21.081 | INFO     | analytics.signal_extraction:run_pipeline:313 - ======================================================================
2026-02-28 23:22:21.081 | INFO     | analytics.signal_extraction:run_pipeline:316 - [1/5] Extracting text units from database...


🚀 Starting Stage 1: NLP Signal Extraction


2026-02-28 23:22:21.222 | INFO     | analytics.signal_extraction:extract_text_units_from_db:88 - Retrieved 10 posts from database
2026-02-28 23:22:21.222 | INFO     | analytics.signal_extraction:extract_text_units_from_db:96 - Extracted 94 text units
2026-02-28 23:22:21.222 | INFO     | analytics.signal_extraction:run_pipeline:329 - [2/5] Preprocessing text units...
2026-02-28 23:22:21.224 | INFO     | analytics.signal_extraction:run_pipeline:334 - Valid text units: 58/94
2026-02-28 23:22:21.224 | INFO     | analytics.signal_extraction:run_pipeline:337 - [3/5] Running NLP analysis (sentiment, emotion, distortion)...
2026-02-28 23:22:21.225 | INFO     | analytics.signal_extraction:_load_nlp_pipeline:57 - Loading NLP models...
2026-02-28 23:22:21.225 | INFO     | analytics.nlp_models:__init__:373 - Initializing NLP Pipeline...
2026-02-28 23:22:21.225 | INFO     | analytics.nlp_models:__init__:33 - Loading sentiment model: cardiffnlp/twitter-roberta-base-sentiment-latest
Loading weights: 


📊 Stage 1 Results:
   • Text units found: 94
   • Valid units analyzed: 58
   • Signals created: 58
   • CSV exported: nlp_signals_output.csv
   • Duration: 16.94s


## Step 4: Inspect NLP Signals

Let's examine some of the signals we just created.

In [71]:
async def inspect_signals(limit=5):
    """View sample signals from database"""
    db = MongoDB.get_db()
    signals_collection = db.text_units_signals
    
    signals = await signals_collection.find().limit(limit).to_list(length=None)
    
    print("=" * 70)
    print(f"🔍 Inspecting {len(signals)} Sample Signals")
    print("=" * 70)
    
    for i, signal in enumerate(signals, 1):
        print(f"\n[{i}] Case User: {signal['case_user']}")
        print(f"    Text: {signal['text'][:80]}...")
        print(f"    Sentiment: {signal['sentiment_label']} ({signal['sentiment_score']:.2f})")
        print(f"    Emotion: {signal['emotion_label']} (distress: {signal['is_distress']})")
        print(f"    Distortion: {'Yes' if signal['distortion_indicator'] else 'No'}")
        if signal['distortion_category']:
            print(f"    → Type: {signal['distortion_category']}")
    
    print("\n" + "=" * 70)

await inspect_signals(limit=5)

🔍 Inspecting 5 Sample Signals

[1] Case User: chrishemsworth
    Text: I’m launching a new series with my best mates! I’m sending @azzagrist and @zocob...
    Sentiment: positive (0.96)
    Emotion: joy (distress: False)
    Distortion: No

[2] Case User: chrishemsworth
    Text: Te invito a mi casa, no tengo mucho, pero descansaras. Chile...
    Sentiment: neutral (0.09)
    Emotion: neutral (distress: False)
    Distortion: No

[3] Case User: chrishemsworth
    Text: Excellence, dedication, and timeless charisma. 👏...
    Sentiment: positive (0.95)
    Emotion: neutral (distress: False)
    Distortion: No

[4] Case User: chrishemsworth
    Text: Health and wellness is an important part of life. Love your body to love you i s...
    Sentiment: positive (0.88)
    Emotion: joy (distress: False)
    Distortion: No

[5] Case User: chrishemsworth
    Text: Take me with you'lls...
    Sentiment: neutral (0.36)
    Emotion: neutral (distress: False)
    Distortion: No



## Step 5: Run Stage 2 - Behavioral Feature Engineering

This stage aggregates signals and computes risk profiles for each case user.

In [72]:
from analytics.feature_engineering import BehavioralFeatureEngineer

async def run_stage2(case_users=None, window_days=7):
    """
    Run Stage 2: Feature Engineering and Risk Scoring
    
    Args:
        case_users: List of specific usernames (None = all with signals)
        window_days: Time window for aggregation (default: 7 days)
    """
    print("🚀 Starting Stage 2: Behavioral Feature Engineering")
    print("=" * 70)
    
    engineer = BehavioralFeatureEngineer()
    
    results = await engineer.run_pipeline(
        case_users=case_users,
        window_days=window_days
    )
    
    return results

# Run Stage 2
stage2_results = await run_stage2(window_days=30)

print("\n📊 Stage 2 Results:")
print(f"   • Case users processed: {stage2_results['case_users_processed']}")
print(f"   • Profiles created: {stage2_results['profiles_created']}")
print(f"   • High-risk cases: {len(stage2_results['high_risk_cases'])}")
if stage2_results['high_risk_cases']:
    print(f"   • ⚠️  High-risk users: {', '.join(stage2_results['high_risk_cases'])}")
print(f"   • Duration: {stage2_results['duration_seconds']:.2f}s")

2026-02-28 23:22:38.129 | INFO     | analytics.feature_engineering:__init__:38 - Behavioral Feature Engineer initialized
2026-02-28 23:22:38.130 | INFO     | analytics.feature_engineering:run_pipeline:469 - ======================================================================
2026-02-28 23:22:38.130 | INFO     | analytics.feature_engineering:run_pipeline:470 - Starting Behavioral Feature Engineering Pipeline (Stage 2)
2026-02-28 23:22:38.131 | INFO     | analytics.feature_engineering:run_pipeline:471 - ======================================================================
2026-02-28 23:22:38.131 | INFO     | analytics.feature_engineering:run_pipeline:475 - Fetching all case users from signals...
2026-02-28 23:22:38.216 | INFO     | analytics.feature_engineering:run_pipeline:478 - Processing 21 case users...
2026-02-28 23:22:38.217 | INFO     | analytics.feature_engineering:run_pipeline:485 - [1/21] Processing a24...
2026-02-28 23:22:38.217 | INFO     | analytics.feature_engineering:co

🚀 Starting Stage 2: Behavioral Feature Engineering


2026-02-28 23:22:38.325 | SUCCESS  | analytics.feature_engineering:compute_risk_profile:424 - Risk profile computed for a24: Score=7.3, Level=Low
2026-02-28 23:22:38.410 | INFO     | analytics.feature_engineering:store_risk_profile:451 - Risk profile stored: updated
2026-02-28 23:22:38.411 | INFO     | analytics.feature_engineering:run_pipeline:485 - [2/21] Processing amazonalexa...
2026-02-28 23:22:38.411 | INFO     | analytics.feature_engineering:compute_risk_profile:359 - Computing risk profile for amazonalexa (window: 30 days)
2026-02-28 23:22:38.480 | SUCCESS  | analytics.feature_engineering:compute_risk_profile:424 - Risk profile computed for amazonalexa: Score=24.7, Level=Low
2026-02-28 23:22:38.554 | INFO     | analytics.feature_engineering:store_risk_profile:451 - Risk profile stored: updated
2026-02-28 23:22:38.554 | INFO     | analytics.feature_engineering:run_pipeline:485 - [3/21] Processing astartingpoint...
2026-02-28 23:22:38.555 | INFO     | analytics.feature_engineerin


📊 Stage 2 Results:
   • Case users processed: 21
   • Profiles created: 21
   • High-risk cases: 0
   • Duration: 4.82s


## Step 6: View Risk Profiles

Let's examine the computed risk profiles and see the prioritization.

In [73]:
async def view_risk_profiles(limit=10):
    """View risk profiles sorted by risk score"""
    db = MongoDB.get_db()
    profiles_collection = db.case_risk_profiles
    
    # Get profiles sorted by risk score (highest first)
    profiles = await profiles_collection.find() \
        .sort('risk_score', -1) \
        .limit(limit) \
        .to_list(length=None)
    
    print("=" * 70)
    print(f"🎯 Top {len(profiles)} Risk Profiles (by Risk Score)")
    print("=" * 70)
    
    for i, profile in enumerate(profiles, 1):
        risk_level = profile['risk_level']
        priority = profile['priority']
        
        # Color coding based on risk
        emoji = "🔴" if risk_level == "High" else "🟡" if risk_level == "Medium" else "🟢"
        
        print(f"\n{emoji} [{i}] {profile['case_user']}")
        print(f"    Risk Score: {profile['risk_score']:.1f}/100")
        print(f"    Level: {risk_level} (Priority {priority})")
        print(f"    Window: {profile['analysis_window_days']} days")
        print(f"    Signals: {profile['total_text_units']} units")
        print(f"    Distortion Rate: {profile['distortion_rate']:.1%}")
        print(f"    Distress Rate: {profile['distress_emotion_rate']:.1%}")
        print(f"    Avg Sentiment: {profile['avg_sentiment_score']:.2f}")
        
        if profile['key_signals']:
            print(f"    Key Signals:")
            for signal in profile['key_signals'][:3]:
                print(f"      • {signal}")
    
    print("\n" + "=" * 70)
    
    return profiles

risk_profiles = await view_risk_profiles(limit=10)

🎯 Top 10 Risk Profiles (by Risk Score)

🟢 [1] crime101film
    Risk Score: 35.6/100
    Level: Low (Priority 3)
    Window: 30 days
    Signals: 33 units
    Distortion Rate: 0.0%
    Distress Rate: 24.2%
    Avg Sentiment: 0.50
    Key Signals:
      • High sentiment volatility: σ=0.59

🟢 [2] astartingpoint
    Risk Score: 35.3/100
    Level: Low (Priority 3)
    Window: 30 days
    Signals: 7 units
    Distortion Rate: 0.0%
    Distress Rate: 42.9%
    Avg Sentiment: 0.13
    Key Signals:
      • High sentiment volatility: σ=0.91
      • Elevated distress emotions: 42.9%
      • High negative sentiment rate: 42.9%

🟢 [3] metsaryhma
    Risk Score: 35.0/100
    Level: Low (Priority 3)
    Window: 30 days
    Signals: 7 units
    Distortion Rate: 0.0%
    Distress Rate: 42.9%
    Avg Sentiment: -0.16
    Key Signals:
      • High sentiment volatility: σ=0.79
      • Elevated distress emotions: 42.9%
      • High negative sentiment rate: 57.1%

🟢 [4] chrishemsworth
    Risk Score: 31.0/

## Step 7: Detailed Case Analysis

Let's do a deep dive into one high-risk case.

In [74]:
async def analyze_case(username):
    """Detailed analysis of a specific case"""
    db = MongoDB.get_db()
    
    # Get risk profile
    profile = await db.case_risk_profiles.find_one({'case_user': username})
    
    if not profile:
        print(f"❌ No risk profile found for {username}")
        return
    
    # Get signals
    signals = await db.text_units_signals.find({'case_user': username}) \
        .sort('processed_at', -1) \
        .to_list(length=None)
    
    print("=" * 70)
    print(f"📋 Detailed Case Analysis: {username}")
    print("=" * 70)
    
    # Risk overview
    print(f"\n🎯 RISK ASSESSMENT")
    print(f"   Score: {profile['risk_score']:.1f}/100")
    print(f"   Level: {profile['risk_level']}")
    print(f"   Priority: {profile['priority']}")
    
    # Signal breakdown
    print(f"\n📊 SIGNAL BREAKDOWN")
    print(f"   Total text units: {profile['total_text_units']}")
    print(f"   Comments received: {profile['total_comments_received']}")
    print(f"   Captions: {profile['total_captions']}")
    
    # Distortion analysis
    print(f"\n🧠 COGNITIVE DISTORTIONS")
    print(f"   Count: {profile['distortion_count']}")
    print(f"   Rate: {profile['distortion_rate']:.1%}")
    if profile['distortion_categories']:
        print(f"   Categories:")
        for cat, count in profile['distortion_categories'].items():
            print(f"      • {cat}: {count}")
    
    # Sentiment analysis
    print(f"\n😊 SENTIMENT ANALYSIS")
    print(f"   Average: {profile['avg_sentiment_score']:.2f}")
    print(f"   Volatility (σ): {profile['sentiment_std']:.3f}")
    print(f"   Negative rate: {profile['negative_sentiment_rate']:.1%}")
    print(f"   Positive rate: {profile['positive_sentiment_rate']:.1%}")
    
    # Emotion analysis
    print(f"\n😢 EMOTION ANALYSIS")
    print(f"   Distress emotions: {profile['distress_emotion_count']}")
    print(f"   Distress rate: {profile['distress_emotion_rate']:.1%}")
    print(f"   Avg distress score: {profile['avg_distress_score']:.3f}")
    if profile['emotion_distribution']:
        print(f"   Distribution:")
        for emotion, count in sorted(profile['emotion_distribution'].items(), 
                                    key=lambda x: x[1], reverse=True):
            print(f"      • {emotion}: {count}")
    
    # Key signals
    print(f"\n⚠️  KEY SIGNALS")
    for signal in profile['key_signals']:
        print(f"   • {signal}")
    
    # Sample concerning comments
    print(f"\n💬 TOP DISTRESS COMMENTS")
    for i, comment in enumerate(profile['top_distress_comments'][:5], 1):
        print(f"   [{i}] \"{comment}\"")
    
    print("\n" + "=" * 70)

# Analyze the first high-risk user (if any)
if risk_profiles and len(risk_profiles) > 0:
    await analyze_case(risk_profiles[0]['case_user'])
else:
    print("No risk profiles available for detailed analysis")

📋 Detailed Case Analysis: crime101film

🎯 RISK ASSESSMENT
   Score: 35.6/100
   Level: Low
   Priority: 3

📊 SIGNAL BREAKDOWN
   Total text units: 33
   Comments received: 27
   Captions: 6

🧠 COGNITIVE DISTORTIONS
   Count: 0
   Rate: 0.0%

😊 SENTIMENT ANALYSIS
   Average: 0.50
   Volatility (σ): 0.590
   Negative rate: 18.2%
   Positive rate: 60.6%

😢 EMOTION ANALYSIS
   Distress emotions: 8
   Distress rate: 24.2%
   Avg distress score: 0.291
   Distribution:
      • neutral: 11
      • joy: 8
      • fear: 6
      • surprise: 5
      • sadness: 2
      • disgust: 1

⚠️  KEY SIGNALS
   • High sentiment volatility: σ=0.59

💬 TOP DISTRESS COMMENTS
   [1] "Just came back from seeing the movie.  My Gawd!! Chris' hair and eyebrows were dyed dark brown!!!..."
   [2] "Just came back from seeing the movie.  My Gawd!! Chris' hair and eyebrows were dyed dark brown!!!..."
   [3] "I love acting but unfortunately I'm a Zimbabwean"
   [4] "I love acting but unfortunately I'm a Zimbabwean"
   [5] 

## Step 8: Export Results for Dashboard

Export the risk profiles in a format ready for dashboard integration.

In [75]:
import json

async def export_for_dashboard(output_file='dashboard_cases.json'):
    """Export risk profiles in dashboard-ready format"""
    db = MongoDB.get_db()
    
    # Get all profiles sorted by priority
    profiles = await db.case_risk_profiles.find() \
        .sort([('priority', 1), ('risk_score', -1)]) \
        .to_list(length=None)
    
    dashboard_cases = []
    
    for profile in profiles:
        case = {
            'username': profile['case_user'],
            'riskLevel': int(profile['risk_score'] / 20) + 1,  # Convert to 1-5 scale
            'riskScore': round(profile['risk_score'], 1),
            'priority': profile['risk_level'],
            'signals': profile['key_signals'],
            'concerningComments': profile['top_distress_comments'][:3],
            'metrics': {
                'distortionRate': round(profile['distortion_rate'] * 100, 1),
                'distressRate': round(profile['distress_emotion_rate'] * 100, 1),
                'sentimentScore': round(profile['avg_sentiment_score'], 2),
                'volatility': round(profile['sentiment_std'], 2)
            },
            'analysisWindow': f"{profile['analysis_window_days']} days",
            'lastUpdated': profile['last_updated'].isoformat()
        }
        dashboard_cases.append(case)
    
    # Write to JSON file
    with open(output_file, 'w') as f:
        json.dump(dashboard_cases, f, indent=2)
    
    print(f"✅ Exported {len(dashboard_cases)} cases to {output_file}")
    print(f"   • High priority: {sum(1 for c in dashboard_cases if c['priority'] == 'High')}")
    print(f"   • Medium priority: {sum(1 for c in dashboard_cases if c['priority'] == 'Medium')}")
    print(f"   • Low priority: {sum(1 for c in dashboard_cases if c['priority'] == 'Low')}")
    
    return dashboard_cases

exported_cases = await export_for_dashboard()

✅ Exported 21 cases to dashboard_cases.json
   • High priority: 0
   • Medium priority: 0
   • Low priority: 21


## Step 9: Pipeline Statistics

View overall analytics statistics.

In [76]:
async def show_pipeline_stats():
    """Display overall pipeline statistics"""
    db = MongoDB.get_db()
    
    # Count documents
    posts_count = await db.instagram_posts.count_documents({})
    signals_count = await db.text_units_signals.count_documents({})
    profiles_count = await db.case_risk_profiles.count_documents({})
    
    # Risk level breakdown
    high_risk = await db.case_risk_profiles.count_documents({'risk_level': 'High'})
    medium_risk = await db.case_risk_profiles.count_documents({'risk_level': 'Medium'})
    low_risk = await db.case_risk_profiles.count_documents({'risk_level': 'Low'})
    
    # Average risk score
    pipeline = [
        {'$group': {
            '_id': None,
            'avg_risk': {'$avg': '$risk_score'},
            'max_risk': {'$max': '$risk_score'},
            'avg_distortion': {'$avg': '$distortion_rate'},
            'avg_distress': {'$avg': '$distress_emotion_rate'}
        }}
    ]
    stats = await db.case_risk_profiles.aggregate(pipeline).to_list(length=1)
    
    print("=" * 70)
    print("📊 ANALYTICS PIPELINE STATISTICS")
    print("=" * 70)
    
    print(f"\n📦 DATA VOLUMES")
    print(f"   Instagram posts: {posts_count:,}")
    print(f"   NLP signals: {signals_count:,}")
    print(f"   Risk profiles: {profiles_count:,}")
    
    print(f"\n🎯 RISK DISTRIBUTION")
    print(f"   🔴 High risk: {high_risk} ({high_risk/profiles_count*100:.1f}%)")
    print(f"   🟡 Medium risk: {medium_risk} ({medium_risk/profiles_count*100:.1f}%)")
    print(f"   🟢 Low risk: {low_risk} ({low_risk/profiles_count*100:.1f}%)")
    
    if stats:
        s = stats[0]
        print(f"\n📈 AVERAGE METRICS")
        print(f"   Risk score: {s['avg_risk']:.1f}/100")
        print(f"   Max risk score: {s['max_risk']:.1f}/100")
        print(f"   Distortion rate: {s['avg_distortion']:.1%}")
        print(f"   Distress rate: {s['avg_distress']:.1%}")
    
    print("\n" + "=" * 70)

await show_pipeline_stats()

📊 ANALYTICS PIPELINE STATISTICS

📦 DATA VOLUMES
   Instagram posts: 164
   NLP signals: 283
   Risk profiles: 21

🎯 RISK DISTRIBUTION
   🔴 High risk: 0 (0.0%)
   🟡 Medium risk: 0 (0.0%)
   🟢 Low risk: 21 (100.0%)

📈 AVERAGE METRICS
   Risk score: 18.5/100
   Max risk score: 35.6/100
   Distortion rate: 0.0%
   Distress rate: 18.3%



## Summary

This notebook demonstrated the complete analytics pipeline:

1. ✅ **NLP Model Testing** - Verified sentiment, emotion, and distortion detection
2. ✅ **Stage 1 Execution** - Extracted and analyzed text signals
3. ✅ **Stage 2 Execution** - Computed behavioral features and risk scores
4. ✅ **Case Analysis** - Reviewed individual high-risk cases
5. ✅ **Dashboard Export** - Prepared data for visualization

### Next Steps

- **API Integration**: Use the REST API endpoints at `/api/analytics/*`
- **Scheduled Processing**: Set up cron jobs to run the pipeline regularly
- **Dashboard Integration**: Connect the frontend to display risk profiles
- **Alert System**: Implement notifications for high-risk cases
- **Model Tuning**: Adjust thresholds and weights based on validation

### API Endpoints Available

- `POST /api/analytics/extract-signals` - Run Stage 1
- `POST /api/analytics/compute-risk-profiles` - Run Stage 2
- `POST /api/analytics/run-full-pipeline` - Run both stages
- `GET /api/analytics/risk-profiles` - Query risk profiles
- `GET /api/analytics/signals/{username}` - Get user signals
- `GET /api/analytics/stats` - Get overall statistics

## Stage 2B: PCA-Based Risk Scoring with LLM Calibration (NEW)

This is the new PCA-based scoring system that:
- Uses **PCA to learn weights** from data (instead of hand-coded weights)
- Applies **low-data damping** via sigmoid function
- Uses **guardrails** to prevent underweighting severe distortions
- Optionally uses **LLM calibration** for bounded ±0.10 adjustment
- Produces **final_score (0-1)** with priority levels: low/medium/high/critical

In [77]:
# Import PCA scoring module
from analytics.stage2_pca_llm import run_case_scoring

print("✓ PCA scoring module loaded")

✓ PCA scoring module loaded


### Run PCA Scoring (Without LLM - Fast)

First, let's run without LLM calibration for faster testing:

In [78]:
# Get database connection for PCA scoring
db = MongoDB.get_db()
print("✓ Database connection ready")

✓ Database connection ready


In [79]:
# CLEAR OLD SIGNALS AND REGENERATE
# ⚠️ WARNING: This will delete all existing signals and regenerate them
# Only run this if the diagnostic shows missing PCA fields

CONFIRM_DELETE = True  # Change to True to actually delete

if CONFIRM_DELETE:
    print("🗑️  Clearing old signals...")
    result = await db.text_units_signals.delete_many({})
    print(f"   Deleted {result.deleted_count} old signals")
    
    print("\n🔄 Re-running Stage 1 to generate fresh signals...")
    stage1_results = await run_stage1(limit=None)  # Process ALL posts
    
    print(f"\n✅ DONE!")
    print(f"   Text units: {stage1_results['text_units_found']}")
    print(f"   Signals created: {stage1_results['signals_created']}")
    print(f"   Duration: {stage1_results['duration_seconds']:.1f}s")
    
    # Verify new fields
    signals_with_fields = await db.text_units_signals.count_documents({
        'distress_emotion': {'$exists': True}
    })
    print(f"\n✓ Signals with PCA fields: {signals_with_fields}")
    print("\nNow scroll down and run the PCA scoring cell again!")
else:
    print("⚠️  CONFIRM_DELETE is False")
    print("To clear old signals and regenerate:")
    print("1. Set CONFIRM_DELETE = True above")
    print("2. Re-run this cell")
    print("\nThis will:")
    print("  • Delete all existing signals")
    print("  • Re-run Stage 1 NLP extraction") 
    print("  • Generate fresh signals with new PCA fields")

2026-02-28 23:22:44.277 | INFO     | analytics.signal_extraction:__init__:52 - NLP Signal Extractor initialized
2026-02-28 23:22:44.277 | INFO     | analytics.signal_extraction:run_pipeline:311 - ======================================================================
2026-02-28 23:22:44.278 | INFO     | analytics.signal_extraction:run_pipeline:312 - Starting NLP Signal Extraction Pipeline (Stage 1)
2026-02-28 23:22:44.278 | INFO     | analytics.signal_extraction:run_pipeline:313 - ======================================================================
2026-02-28 23:22:44.278 | INFO     | analytics.signal_extraction:run_pipeline:316 - [1/5] Extracting text units from database...


🗑️  Clearing old signals...
   Deleted 283 old signals

🔄 Re-running Stage 1 to generate fresh signals...
🚀 Starting Stage 1: NLP Signal Extraction


2026-02-28 23:22:45.199 | INFO     | analytics.signal_extraction:extract_text_units_from_db:88 - Retrieved 164 posts from database
2026-02-28 23:22:45.201 | INFO     | analytics.signal_extraction:extract_text_units_from_db:96 - Extracted 326 text units
2026-02-28 23:22:45.202 | INFO     | analytics.signal_extraction:run_pipeline:329 - [2/5] Preprocessing text units...
2026-02-28 23:22:45.209 | INFO     | analytics.signal_extraction:run_pipeline:334 - Valid text units: 225/326
2026-02-28 23:22:45.209 | INFO     | analytics.signal_extraction:run_pipeline:337 - [3/5] Running NLP analysis (sentiment, emotion, distortion)...
2026-02-28 23:22:45.210 | INFO     | analytics.signal_extraction:_load_nlp_pipeline:57 - Loading NLP models...
2026-02-28 23:22:45.210 | INFO     | analytics.nlp_models:__init__:373 - Initializing NLP Pipeline...
2026-02-28 23:22:45.210 | INFO     | analytics.nlp_models:__init__:33 - Loading sentiment model: cardiffnlp/twitter-roberta-base-sentiment-latest
Loading weigh


✅ DONE!
   Text units: 326
   Signals created: 225
   Duration: 37.2s

✓ Signals with PCA fields: 225

Now scroll down and run the PCA scoring cell again!


In [80]:
# RELOAD MODULES - Run this before regenerating signals
# This picks up the bug fix in text_preprocessing.py

import importlib
import analytics.text_preprocessing
import analytics.signal_extraction

importlib.reload(analytics.text_preprocessing)
importlib.reload(analytics.signal_extraction)

from analytics.signal_extraction import NLPSignalExtractor

print("✅ Modules reloaded with latest bug fixes")

✅ Modules reloaded with latest bug fixes


### Fix: Clear Old Signals and Regenerate

If the diagnostic shows signals without new fields, you need to clear them and re-run Stage 1:

In [81]:
# Run PCA scoring without LLM (fast)
import time

start_time = time.time()

pca_results = await run_case_scoring(
    db=db,
    window_days=30,  # Analyze last 30 days
    limit_users=None,  # Process all users (or set to 5 for testing)
    use_llm=False  # Skip LLM for faster testing
)

duration = time.time() - start_time

print(f"\n{'='*60}")
print(f"PCA Scoring Results (No LLM)")
print(f"{'='*60}")
print(f"Success: {pca_results['success']}")
print(f"Users processed: {pca_results['n_profiles']}")
print(f"PCA run ID: {pca_results.get('pca_run_id', 'N/A')}")
print(f"Duration: {duration:.2f} seconds")

if pca_results['success'] and pca_results['n_profiles'] > 0:
    print(f"\n{'='*60}")
    print("Sample Results (Top 3 Users by Risk)")
    print(f"{'='*60}")
    
    # Sort by final_score
    sorted_profiles = sorted(pca_results['profiles'], key=lambda x: x['final_score'], reverse=True)
    
    for i, profile in enumerate(sorted_profiles[:3], 1):
        print(f"\n{i}. User: {profile['username']}")
        print(f"   Text units analyzed: {profile['n_units']}")
        print(f"   Damping factor: {profile['damp']:.3f}")
        print(f"   ---")
        print(f"   Emotion score:    {profile['emotion_score']:.3f}")
        print(f"   Sentiment score:  {profile['sentiment_score']:.3f}")
        print(f"   Harm score:       {profile['harm_score']:.3f}")
        print(f"   ---")
        print(f"   Risk (PCA):       {profile['risk_score_math']:.3f}")
        print(f"   Base score:       {profile['base_score']:.3f}")
        print(f"   LLM delta:        {profile['llm_delta']:+.3f}")
        print(f"   ---")
        print(f"   ⭐ FINAL SCORE:   {profile['final_score']:.3f}")
        print(f"   Priority:         {profile['priority_level'].upper()}")
        print(f"   Evidence units:   {len(profile['evidence_unit_ids'])}")
        
        # Show PCA loadings for first user
        if i == 1:
            loadings = profile['pc1_loadings']
            print(f"\n   PCA Learned Weights:")
            print(f"     Emotion:   {loadings['emotion']:.3f}")
            print(f"     Sentiment: {loadings['sentiment']:.3f}")
            print(f"     Harm:      {loadings['harm']:.3f}")

2026-02-28 23:23:21.583 | INFO     | analytics.stage2_pca_llm:run_case_scoring:332 - Running Stage 2 PCA scoring (window=30 days)...
2026-02-28 23:23:21.650 | INFO     | analytics.stage2_pca_llm:run_case_scoring:348 - Found 20 users with signals
2026-02-28 23:23:21.650 | INFO     | analytics.stage2_pca_llm:run_case_scoring:359 - Aggregating per-user signals...
2026-02-28 23:23:23.022 | INFO     | analytics.stage2_pca_llm:run_case_scoring:366 - Aggregated 20 users
2026-02-28 23:23:23.067 | INFO     | analytics.stage2_pca_llm:run_case_scoring:378 - Running PCA (run_id=7bb5d36d)...
2026-02-28 23:23:23.160 | INFO     | analytics.stage2_pca_llm:run_pca_scoring:219 - PCA loadings: emotion=-0.707, sentiment=0.707, harm=-0.000
2026-02-28 23:23:23.163 | INFO     | analytics.stage2_pca_llm:run_case_scoring:382 - Applying guardrails and LLM calibration...
2026-02-28 23:23:23.163 | INFO     | analytics.stage2_pca_llm:run_case_scoring:385 - Processing user 1/20: chrishemsworth
2026-02-28 23:23:23.1


PCA Scoring Results (No LLM)
Success: True
Users processed: 20
PCA run ID: 7bb5d36d
Duration: 2.99 seconds

Sample Results (Top 3 Users by Risk)

1. User: mrbeast
   Text units analyzed: 1
   Damping factor: 0.088
   ---
   Emotion score:    0.002
   Sentiment score:  0.088
   Harm score:       0.000
   ---
   Risk (PCA):       1.000
   Base score:       1.000
   LLM delta:        +0.000
   ---
   ⭐ FINAL SCORE:   1.000
   Priority:         CRITICAL
   Evidence units:   1

   PCA Learned Weights:
     Emotion:   -0.707
     Sentiment: 0.707
     Harm:      -0.000

2. User: mohammed.usrof
   Text units analyzed: 1
   Damping factor: 0.088
   ---
   Emotion score:    0.067
   Sentiment score:  0.088
   Harm score:       0.000
   ---
   Risk (PCA):       0.864
   Base score:       0.864
   LLM delta:        +0.000
   ---
   ⭐ FINAL SCORE:   0.864
   Priority:         CRITICAL
   Evidence units:   1

3. User: metsaryhma
   Text units analyzed: 1
   Damping factor: 0.088
   ---
   Emotion 

### Compare Legacy vs PCA Scoring

Let's compare the old hand-coded scores with the new PCA scores:

In [82]:
# Compare legacy vs PCA scores
if pca_results['success'] and pca_results['n_profiles'] > 0:
    print(f"{'='*80}")
    print(f"Legacy (Hand-coded) vs PCA (Learned Weights) Comparison")
    print(f"{'='*80}")
    
    # Check if we have the legacy profiles from earlier in the notebook
    if 'risk_profiles' in dir() and risk_profiles:
        print(f"Using in-memory legacy profiles from Stage 2A")
        print(f"{'Username':<20} {'Legacy Score':<15} {'PCA Score':<12} {'Priority':<12} {'Change':<10}")
        print(f"{'-'*80}")
        
        # Create dict from in-memory legacy profiles
        legacy_dict = {p['case_user']: p for p in risk_profiles}
        
        for profile in sorted(pca_results['profiles'], key=lambda x: x['final_score'], reverse=True):
            username = profile['username']
            pca_score = profile['final_score']
            priority = profile['priority_level']
            
            # Get legacy score (0-100 scale, convert to 0-1 for comparison)
            legacy = legacy_dict.get(username, {})
            legacy_score = legacy.get('risk_score', 0) / 100.0  # Convert to 0-1
            
            # Calculate change
            change = pca_score - legacy_score
            change_str = f"{change:+.3f}" if legacy_score > 0 else "N/A"
            
            print(f"{username:<20} {legacy_score:.3f} (0-1)   {pca_score:.3f}      {priority:<12} {change_str}")
        
        print(f"\n{'='*80}")
        print("Note: Legacy scores are converted from 0-100 to 0-1 scale for comparison")
        print("Positive change means PCA scores higher risk than legacy system")
    else:
        print("⚠️  No in-memory legacy profiles found!")
        print("You need to run Stage 2A (legacy) first before comparing.")
        print("Scroll up to 'Step 5: Run Stage 2 - Behavioral Feature Engineering' and run that cell.")
        print("\nNote: PCA has overwritten the database profiles, so we can't compare from DB.")
else:
    print("No PCA results to compare")

Legacy (Hand-coded) vs PCA (Learned Weights) Comparison
Using in-memory legacy profiles from Stage 2A
Username             Legacy Score    PCA Score    Priority     Change    
--------------------------------------------------------------------------------
mrbeast              0.000 (0-1)   1.000      critical     N/A
mohammed.usrof       0.307 (0-1)   0.864      critical     +0.557
metsaryhma           0.350 (0-1)   0.841      critical     +0.490
markruffalo          0.236 (0-1)   0.588      high         +0.352
a24                  0.000 (0-1)   0.586      high         N/A
astartingpoint       0.353 (0-1)   0.586      high         +0.233
thinkjinx            0.000 (0-1)   0.586      high         N/A
robertdowneyjr       0.210 (0-1)   0.583      high         +0.373
taylorswift          0.000 (0-1)   0.581      high         N/A
mileycyrus           0.000 (0-1)   0.577      high         N/A
harrystyles          0.000 (0-1)   0.573      high         N/A
disneyplus           0.000 (0-1)   

### (Optional) Run PCA Scoring with LLM Calibration

**Requirements:** 
- Ollama must be installed and running
- Run `ollama serve` in another terminal
- Model downloaded: `ollama pull llama2`

This adds LLM-based calibration with bounded ±0.10 delta adjustment:

In [87]:
# Run PCA scoring WITH LLM calibration (slower)
# Uncomment and run only if you have Ollama installed and running


print("Running PCA scoring with LLM calibration...")
print("This requires Ollama to be running on localhost:11434")
print("")

try:
    llm_results = await run_case_scoring(
        db=db,
        window_days=30,
        limit_users=3,  # Test on 3 users (LLM is slower)
        use_llm=True,
        llm_model="llama2",
        ollama_url="http://localhost:11434"
    )
    
    if llm_results['success']:
        print(f"\n{'='*60}")
        print("LLM Calibration Results")
        print(f"{'='*60}")
        
        for profile in llm_results['profiles']:
            print(f"\nUser: {profile['username']}")
            print(f"  Base score:       {profile['base_score']:.3f}")
            print(f"  LLM delta (run 1): {profile.get('llm_delta1', 0):+.3f}")
            print(f"  LLM delta (run 2): {profile.get('llm_delta2', 0):+.3f}")
            print(f"  LLM delta (avg):   {profile['llm_delta']:+.3f}")
            print(f"  Final score:      {profile['final_score']:.3f}")
            print(f"  Change:           {profile['llm_delta']:+.3f} ({abs(profile['llm_delta']*100):.1f}%)")
    else:
        print(f"LLM calibration failed: {llm_results.get('message')}")
        
except Exception as e:
    print(f"Error: {e}")
    print("Make sure Ollama is installed and running: ollama serve")


print("LLM calibration cell ready (currently commented out)")
print("Uncomment the code above to test LLM calibration")
print("Ensure Ollama is running first: ollama serve")

2026-02-28 23:34:40.289 | INFO     | analytics.stage2_pca_llm:run_case_scoring:332 - Running Stage 2 PCA scoring (window=30 days)...
2026-02-28 23:34:40.361 | INFO     | analytics.stage2_pca_llm:run_case_scoring:348 - Found 3 users with signals
2026-02-28 23:34:40.361 | INFO     | analytics.stage2_pca_llm:run_case_scoring:359 - Aggregating per-user signals...


Running PCA scoring with LLM calibration...
This requires Ollama to be running on localhost:11434



2026-02-28 23:34:40.583 | INFO     | analytics.stage2_pca_llm:run_case_scoring:366 - Aggregated 3 users
2026-02-28 23:34:40.587 | INFO     | analytics.stage2_pca_llm:run_case_scoring:378 - Running PCA (run_id=26ccb9de)...
2026-02-28 23:34:40.604 | INFO     | analytics.stage2_pca_llm:run_pca_scoring:219 - PCA loadings: emotion=0.707, sentiment=0.707, harm=0.000
2026-02-28 23:34:40.605 | INFO     | analytics.stage2_pca_llm:run_case_scoring:382 - Applying guardrails and LLM calibration...
2026-02-28 23:34:40.605 | INFO     | analytics.stage2_pca_llm:run_case_scoring:385 - Processing user 1/3: metsaryhma
2026-02-28 23:34:40.606 | INFO     | analytics.llm_calibration:calibrate_score:171 - Running LLM calibration for metsaryhma (2 runs)...
2026-02-28 23:35:10.652 | ERROR    | analytics.llm_calibration:_call_ollama:145 - LLM call failed: 
2026-02-28 23:35:10.654 | WARNING  | analytics.llm_calibration:calibrate_score:186 -   Run 1: failed (using 0.0)
2026-02-28 23:35:16.512 | INFO     | analyt


LLM Calibration Results

User: metsaryhma
  Base score:       1.000
  LLM delta (run 1): +0.000
  LLM delta (run 2): +0.010
  LLM delta (avg):   +0.005
  Final score:      1.000
  Change:           +0.005 (0.5%)

User: mohammed.usrof
  Base score:       0.925
  LLM delta (run 1): +0.010
  LLM delta (run 2): +0.010
  LLM delta (avg):   +0.010
  Final score:      0.935
  Change:           +0.010 (1.0%)

User: markruffalo
  Base score:       0.000
  LLM delta (run 1): +0.010
  LLM delta (run 2): +0.010
  LLM delta (avg):   +0.010
  Final score:      0.010
  Change:           +0.010 (1.0%)
LLM calibration cell ready (currently commented out)
Uncomment the code above to test LLM calibration
Ensure Ollama is running first: ollama serve


### Verify MongoDB Storage

Check that PCA profiles are stored in the database:

In [84]:
# Verify PCA profiles in MongoDB
latest_profile = await db.case_risk_profiles.find_one(
    {'final_score': {'$exists': True}},  # Has PCA fields
    sort=[('timestamp', -1)]
)

if latest_profile:
    print(f"{'='*60}")
    print("Latest PCA Profile in Database")
    print(f"{'='*60}")
    print(f"User: {latest_profile['case_user']}")
    print(f"Timestamp: {latest_profile['timestamp']}")
    print(f"Window: {latest_profile.get('window_days', 'N/A')} days")
    print(f"\nPCA Fields Present:")
    
    pca_fields = [
        'emotion_score', 'sentiment_score', 'harm_score',
        'n_units', 'damp', 'risk_score_math', 'base_score',
        'llm_delta', 'final_score', 'priority_level', 'pc1_loadings'
    ]
    
    for field in pca_fields:
        if field in latest_profile:
            value = latest_profile[field]
            if isinstance(value, float):
                print(f"  ✓ {field:<18} = {value:.3f}")
            elif isinstance(value, dict):
                print(f"  ✓ {field:<18} = {value}")
            else:
                print(f"  ✓ {field:<18} = {value}")
        else:
            print(f"  ✗ {field:<18} = MISSING")
    
    print(f"\nEvidence unit IDs: {len(latest_profile.get('evidence_unit_ids', []))} tracked")
else:
    print("No PCA profiles found in database yet")
    print("Run the PCA scoring cell above first")

Latest PCA Profile in Database
User: crime101film
Timestamp: 2026-02-28 23:23:24.502000
Window: 30 days

PCA Fields Present:
  ✓ emotion_score      = 0.189
  ✓ sentiment_score    = 0.000
  ✓ harm_score         = 0.000
  ✓ n_units            = 4
  ✓ damp               = 0.209
  ✓ risk_score_math    = 0.195
  ✓ base_score         = 0.195
  ✓ llm_delta          = 0.000
  ✓ final_score        = 0.195
  ✓ priority_level     = low
  ✓ pc1_loadings       = {'emotion': -0.7071067811865462, 'sentiment': 0.7071067811865488, 'harm': -0.0}

Evidence unit IDs: 4 tracked
